In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Dict

# =============================================================================
# 0) Configuration block (2x2 grid)
# =============================================================================
PLOT_CFG = {
    "height": 2.5,
    "aspect": 1.5,
    "use_dashed": True,
    "linewidth": 2.0,
    "shade_alpha": 0.18,
    "style": "whitegrid",
    "palette": "deep",
    "dpi": 150,
    "xlabel": "Training Steps",
    "ylabel": "Accuracy (%)",
    "legend_fontsize": 9,
    "outfile": "result_single_vs_repeated_2x2.pdf",
}

def make_title_map(models):
    """Map model names to SINGLE / REPEATED using prefix matching."""
    d = {}
    for m in models:
        if m.startswith("sec31_base"):
            d[m] = "SINGLE"
        elif m.startswith("sec31_multi"):
            d[m] = "REPEATED"
    return d

METRIC_MAP: Dict[str, str] = {
    'em/param':             r'$\mathrm{Acc}_{\mathrm{PKU}}$',
    'em/multi_ood_in_ctx':  r'$\mathrm{Acc}_{\mathrm{ICKU}}$',
    'em/pert_ctx_orig':     r'$\mathrm{Pref}_{\mathrm{PK}}$',
    'em/pert_ctx_pert':     r'$\mathrm{Pref}_{\mathrm{ICK}}$',
}

COLOR_MAP = {
    r'$\mathrm{Acc}_{\mathrm{PKU}}$': sns.color_palette("deep")[2],
    r'$\mathrm{Acc}_{\mathrm{ICKU}}$': sns.color_palette("deep")[0],
    r'$\mathrm{Pref}_{\mathrm{PK}}$':  sns.color_palette("deep")[3],
    r'$\mathrm{Pref}_{\mathrm{ICK}}$': sns.color_palette("deep")[1],
}

LINESTYLE_MAP = {
    r'$\mathrm{Acc}_{\mathrm{PKU}}$': (2, 2),
    r'$\mathrm{Acc}_{\mathrm{ICKU}}$': (2, 2),
    r'$\mathrm{Pref}_{\mathrm{PK}}$':  (1, 1),
    r'$\mathrm{Pref}_{\mathrm{ICK}}$': (1, 1),
}

# =============================================================================
# 1) Style setup
# =============================================================================
def setup_style() -> None:
    sns.set_theme(
        style=PLOT_CFG["style"],
        palette=PLOT_CFG["palette"],
        rc={
            "figure.dpi": PLOT_CFG["dpi"],
            "axes.titlesize": 14,
            "axes.labelsize": 12,
            "legend.fontsize": PLOT_CFG["legend_fontsize"],
            "lines.linewidth": PLOT_CFG["linewidth"],
        },
    )

# =============================================================================
# 2) Data loading & statistics (single run, no seed column)
# =============================================================================
def load_stats(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # Build TITLE_MAP dynamically from model names in CSV
    title_map = make_title_map(df["model"].unique())

    rename_cols = {k: v for k, v in METRIC_MAP.items() if k in df.columns}
    df = df.rename(columns=rename_cols)
    metrics = list(rename_cols.values())

    long = pd.melt(
        df,
        id_vars=["model", "step"],
        value_vars=metrics,
        var_name="Metric",
        value_name="Accuracy",
    )

    if long["Accuracy"].max() <= 1.0:
        long["Accuracy"] *= 100.0

    long["Model_Title"] = long["model"].map(title_map).fillna(long["model"])

    # Single run: Mean = value, Std = 0
    long["Mean"] = long["Accuracy"]
    long["Std"] = 0.0

    stats = (
        long[["model", "Model_Title", "step", "Metric", "Mean", "Std"]]
        .sort_values(["Model_Title", "Metric", "step"])
        .reset_index(drop=True)
    )
    return stats

# =============================================================================
# 3) 2x2 FacetGrid plotting
# =============================================================================
def plot_2x2_facet(stats_df: pd.DataFrame) -> None:
    # Add metric type (Acc / Pref) column
    acc_metrics = [r'$\mathrm{Acc}_{\mathrm{PKU}}$', r'$\mathrm{Acc}_{\mathrm{ICKU}}$']
    pref_metrics = [r'$\mathrm{Pref}_{\mathrm{PK}}$', r'$\mathrm{Pref}_{\mathrm{ICK}}$']
    
    df = stats_df.copy()
    df["MetricType"] = df["Metric"].apply(
        lambda x: "Acc" if x in acc_metrics else ("Pref" if x in pref_metrics else None)
    )
    df = df.dropna(subset=["MetricType"])
    
    # Filter SINGLE, REPEATED only
    df = df[df["Model_Title"].isin(["SINGLE", "REPEATED"])]
    
    # FacetGrid: row=MetricType, col=Model_Title
    g = sns.FacetGrid(
        df,
        row="MetricType",
        col="Model_Title",
        row_order=["Acc", "Pref"],
        col_order=["SINGLE", "REPEATED"],
        sharey="row",
        sharex=True,
        height=PLOT_CFG["height"],
        aspect=PLOT_CFG["aspect"],
        margin_titles=False,
    )

    def facet_plot(data, **kwargs):
        ax = plt.gca()
        for metric, mdf in data.groupby("Metric", sort=False):
            mdf = mdf.sort_values("step")
            ax.plot(
                mdf["step"],
                mdf["Mean"],
                label=metric,
                color=COLOR_MAP[metric],
                dashes=LINESTYLE_MAP[metric],
            )
            ax.fill_between(
                mdf["step"],
                mdf["Mean"] - mdf["Std"],
                mdf["Mean"] + mdf["Std"],
                color=COLOR_MAP[metric],
                alpha=PLOT_CFG["shade_alpha"],
            )

    g.map_dataframe(facet_plot)

    # Title, label, legend settings
    for ax in g.axes.flat:
        ax.grid(alpha=0.3)
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            ax.legend(handles, labels, fontsize=PLOT_CFG["legend_fontsize"], loc="center right", frameon=True)
    
    # Top row titles (SINGLE, REPEATED)
    for ax, title in zip(g.axes[0], ["SINGLE", "REPEATED"]):
        ax.set_title(title, fontsize=13, weight="bold", style="italic")
    
    # Remove titles for bottom row
    for ax in g.axes[1]:
        ax.set_title("")
    
    # Set x, y labels
    for ax in g.axes[1]:
        ax.set_xlabel(PLOT_CFG["xlabel"])
    for ax in g.axes[:, 0]:
        ax.set_ylabel(PLOT_CFG["ylabel"])

    g.fig.tight_layout()
    g.fig.savefig(PLOT_CFG["outfile"], bbox_inches="tight", format="pdf")
    print(f"Saved 2x2 FacetGrid plot as {PLOT_CFG['outfile']}")
    plt.show()

# =============================================================================
# 4) Execution
# =============================================================================
if __name__ == "__main__":
    setup_style()
    df_stats = load_stats("../probe_results.csv")
    plot_2x2_facet(df_stats)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Dict

# =============================================================================
# 0) Configuration block (data/metrics same as original code)
# =============================================================================
PLOT_CFG = {
    "height": 2.0,
    "aspect": 1.7,
    "use_dashed": True,
    "linewidth": 2.0,
    "shade_alpha": 0.18,
    "style": "whitegrid",
    "palette": "deep",
    "dpi": 150,
    "xlabel": "Training Steps",
    "ylabel": "Accuracy (%)",
    "legend_fontsize": 10,
    "outfile": "result_single_vs_repeated.pdf",
}

def make_title_map(models):
    """Map model names to SINGLE / REPEATED using prefix matching."""
    d = {}
    for m in models:
        if m.startswith("sec31_base"):
            d[m] = "SINGLE"
        elif m.startswith("sec31_multi"):
            d[m] = "REPEATED"
    return d

METRIC_MAP: Dict[str, str] = {
    'em/param':             r'$\mathrm{Acc}_{\mathrm{PKU}}$',
    'em/multi_ood_in_ctx':  r'$\mathrm{Acc}_{\mathrm{ICKU}}$',
    'em/pert_ctx_orig':     r'$\mathrm{Pref}_{\mathrm{PK}}$',
    'em/pert_ctx_pert':     r'$\mathrm{Pref}_{\mathrm{ICK}}$',
}

COLOR_MAP = {
    r'$\mathrm{Acc}_{\mathrm{PKU}}$': sns.color_palette("deep")[2],
    r'$\mathrm{Acc}_{\mathrm{ICKU}}$': sns.color_palette("deep")[0],
    r'$\mathrm{Pref}_{\mathrm{PK}}$':  sns.color_palette("deep")[3],
    r'$\mathrm{Pref}_{\mathrm{ICK}}$': sns.color_palette("deep")[1],
}

LINESTYLE_MAP = {
    r'$\mathrm{Acc}_{\mathrm{PKU}}$': (2, 2),
    r'$\mathrm{Acc}_{\mathrm{ICKU}}$': (2, 2),
    r'$\mathrm{Pref}_{\mathrm{PK}}$':  (1, 1),
    r'$\mathrm{Pref}_{\mathrm{ICK}}$': (1, 1),
}

PANEL_TITLE_MAP = {
    "SINGLE": "SINGLE",
    "REPEATED_ACC": "REPEATED",
    "REPEATED_PREF": "REPEATED",
}

# =============================================================================
# 1) Style setup
# =============================================================================
def setup_style() -> None:
    sns.set_theme(
        style=PLOT_CFG["style"],
        palette=PLOT_CFG["palette"],
        rc={
            "figure.dpi": PLOT_CFG["dpi"],
            "axes.titlesize": 14,
            "axes.labelsize": 12,
            "legend.fontsize": PLOT_CFG["legend_fontsize"],
            "lines.linewidth": PLOT_CFG["linewidth"],
        },
    )

# =============================================================================
# 2) Data loading & statistics (single run, no seed column)
# =============================================================================
def load_stats(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # Build TITLE_MAP dynamically from model names in CSV
    title_map = make_title_map(df["model"].unique())

    rename_cols = {k: v for k, v in METRIC_MAP.items() if k in df.columns}
    df = df.rename(columns=rename_cols)
    metrics = list(rename_cols.values())

    long = pd.melt(
        df,
        id_vars=["model", "step"],
        value_vars=metrics,
        var_name="Metric",
        value_name="Accuracy",
    )

    if long["Accuracy"].max() <= 1.0:
        long["Accuracy"] *= 100.0

    long["Model_Title"] = long["model"].map(title_map).fillna(long["model"])

    # Single run: Mean = value, Std = 0
    long["Mean"] = long["Accuracy"]
    long["Std"] = 0.0

    stats = (
        long[["model", "Model_Title", "step", "Metric", "Mean", "Std"]]
        .sort_values(["Model_Title", "Metric", "step"])
        .reset_index(drop=True)
    )
    return stats

# =============================================================================
# 3) Panel column creation
# =============================================================================
def add_panel_column(stats_df: pd.DataFrame) -> pd.DataFrame:
    df = stats_df.copy()
    df["Panel"] = None

    acc_metrics = [r'$\mathrm{Acc}_{\mathrm{PKU}}$', r'$\mathrm{Acc}_{\mathrm{ICKU}}$']
    pref_metrics = [r'$\mathrm{Pref}_{\mathrm{PK}}$', r'$\mathrm{Pref}_{\mathrm{ICK}}$']

    df.loc[(df["Model_Title"] == "SINGLE") &
           (df["Metric"].isin(acc_metrics)), "Panel"] = "SINGLE"

    df.loc[(df["Model_Title"] == "REPEATED") &
           (df["Metric"].isin(acc_metrics)), "Panel"] = "REPEATED_ACC"

    df.loc[(df["Model_Title"] == "REPEATED") &
           (df["Metric"].isin(pref_metrics)), "Panel"] = "REPEATED_PREF"

    return df.dropna(subset=["Panel"])

# =============================================================================
# 4) FacetGrid plotting (individual legend per panel)
# =============================================================================
def plot_three_facet(stats_df: pd.DataFrame) -> None:
    data = add_panel_column(stats_df)

    g = sns.FacetGrid(
        data,
        col="Panel",
        col_order=["SINGLE", "REPEATED_ACC", "REPEATED_PREF"],
        sharey=True,
        height=PLOT_CFG["height"],
        aspect=PLOT_CFG["aspect"],
        margin_titles=False,
    )

    def facet_plot(data, **kwargs):
        ax = plt.gca()

        for metric, mdf in data.groupby("Metric", sort=False):
            mdf = mdf.sort_values("step")

            ax.plot(
                mdf["step"],
                mdf["Mean"],
                label=metric,
                color=COLOR_MAP[metric],
                dashes=LINESTYLE_MAP[metric],
            )

            ax.fill_between(
                mdf["step"],
                mdf["Mean"] - mdf["Std"],
                mdf["Mean"] + mdf["Std"],
                color=COLOR_MAP[metric],
                alpha=PLOT_CFG["shade_alpha"],
            )

    g.map_dataframe(facet_plot)

    # Per-panel legend + style
    for ax, panel_name in zip(g.axes.flat, ["SINGLE", "REPEATED_ACC", "REPEATED_PREF"]):
        ax.set_title(PANEL_TITLE_MAP[panel_name], fontsize=13, weight="bold", style="italic")
        ax.set_xlabel(PLOT_CFG["xlabel"])
        ax.grid(alpha=0.3)

        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles, labels, fontsize=PLOT_CFG["legend_fontsize"], loc="center right", frameon=True)

    g.axes.flat[0].set_ylabel(PLOT_CFG["ylabel"])

    g.fig.tight_layout()
    g.fig.savefig(PLOT_CFG["outfile"], bbox_inches="tight", format="pdf")
    print(f"Saved FacetGrid plot as {PLOT_CFG['outfile']}")
    plt.show()

# =============================================================================
# 5) Execution
# =============================================================================
if __name__ == "__main__":
    setup_style()
    df_stats = load_stats("../probe_results.csv")
    plot_three_facet(df_stats)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Dict, List

# =============================================================================
# 0) Single configuration block
# =============================================================================
PLOT_CFG = {
    "height": 2.0,
    "aspect": 1.7,
    "use_dashed": True,
    "linewidth": 2.0,
    "shade_alpha": 0.18,
    "style": "whitegrid",
    "palette": "deep",
    "dpi": 150,
    "xlabel": "Training Steps",
    "ylabel": "Accuracy (%)",
    "legend_out": True,
    "legend_loc": "center left",
    "legend_bbox": (1.02, 0.5),
    "legend_ncol": 1,
    "legend_fontsize": 11,
    "right_pad": 1,
    "outfile": "result_noise01_only_1percent.pdf",
}

# =============================================================================
# 1) Model name -> panel title (prefix-based matching)
# =============================================================================
def make_title_map(models):
    """Map model names using prefix matching."""
    d = {}
    for m in models:
        if m.startswith("sec31_multi"):
            d[m] = "without noise"
        elif m.startswith("sec32_noise001"):
            d[m] = "1% noise"
    return d

# =============================================================================
# 2) CSV column name -> legend labels
# =============================================================================
METRIC_MAP: Dict[str, str] = {
    'em/param': r'$\mathrm{Acc}_{\mathrm{PKU}}$',
    'em/multi_ood_in_ctx': r'$\mathrm{Acc}_{\mathrm{ICKU}}$',
    'em/pert_ctx_orig': r'$\mathrm{Pref}_{\mathrm{PK}}$',
    'em/pert_ctx_pert': r'$\mathrm{Pref}_{\mathrm{ICK}}$',
}

# =============================================================================
# 3) Color / style
# =============================================================================
COLOR_MAP = {
    r'$\mathrm{Acc}_{\mathrm{PKU}}$': sns.color_palette("deep")[2],
    r'$\mathrm{Acc}_{\mathrm{ICKU}}$': sns.color_palette("deep")[0],
    r'$\mathrm{Pref}_{\mathrm{PK}}$':  sns.color_palette("deep")[3],
    r'$\mathrm{Pref}_{\mathrm{ICK}}$': sns.color_palette("deep")[1],
}

LINESTYLE_MAP = {
    r'$\mathrm{Acc}_{\mathrm{PKU}}$': (2, 2),
    r'$\mathrm{Acc}_{\mathrm{ICKU}}$': (2, 2),
    r'$\mathrm{Pref}_{\mathrm{PK}}$':  (2, 4),
    r'$\mathrm{Pref}_{\mathrm{ICK}}$': (2, 4),
}

# =============================================================================
# 4) Style setup
# =============================================================================
def setup_style() -> None:
    sns.set_theme(
        style=PLOT_CFG["style"],
        palette=PLOT_CFG["palette"],
        rc={
            "figure.dpi": PLOT_CFG["dpi"],
            "axes.titlesize": 14,
            "axes.labelsize": 12,
            "legend.fontsize": PLOT_CFG["legend_fontsize"],
            "lines.linewidth": PLOT_CFG["linewidth"],
        }
    )

# =============================================================================
# 5) Load data (single run, no seed column)
# =============================================================================
def load_stats(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # Build title map dynamically
    title_map = make_title_map(df["model"].unique())

    rename_cols = {k: v for k, v in METRIC_MAP.items() if k in df.columns}
    df = df.rename(columns=rename_cols)

    metrics = list(rename_cols.values())

    long = pd.melt(
        df,
        id_vars=["model", "step"],
        value_vars=metrics,
        var_name="Metric",
        value_name="Accuracy",
    )

    # 0-1 -> % conversion
    if long["Accuracy"].max() <= 1.0:
        long["Accuracy"] *= 100.0

    long["Model_Title"] = long["model"].map(title_map).fillna(long["model"])

    # Single run: Mean = value, Std = 0
    long["Mean"] = long["Accuracy"]
    long["Std"] = 0.0

    stats = (
        long[["model", "Model_Title", "step", "Metric", "Mean", "Std"]]
        .sort_values(["Model_Title", "Metric", "step"])
        .reset_index(drop=True)
    )
    return stats

# =============================================================================
# 6) Plot function (single panel output)
# =============================================================================
def plot_with_std(stats_df: pd.DataFrame, model_titles: List[str], outfile: str = None) -> None:
    # --- Extract only the selected models by title ---
    data = stats_df[stats_df["Model_Title"].isin(model_titles)].copy()

    if data.empty:
        print("No data for selected model")
        return

    # FacetGrid -> only 1 panel created
    g = sns.FacetGrid(
        data,
        col="Model_Title",
        col_order=model_titles,
        height=PLOT_CFG["height"],
        aspect=PLOT_CFG["aspect"],
        sharey=True,
        margin_titles=False
    )

    # --- Line + std shading ---
    for ax, title in zip(g.axes.flat, model_titles):
        sub = data[data["Model_Title"] == title]

        for metric, mdf in sub.groupby("Metric", sort=False):
            mdf = mdf.sort_values("step")
            x, y, s = mdf["step"], mdf["Mean"], mdf["Std"]
            color = COLOR_MAP[metric]

            ax.plot(
                x, y,
                label=metric,
                color=color,
                dashes=LINESTYLE_MAP[metric],
            )
            ax.fill_between(x, y - s, y + s, color=color, alpha=PLOT_CFG["shade_alpha"])

        ax.set_title(title, fontsize=14, weight="bold", style="italic")
        ax.set_xlabel(PLOT_CFG["xlabel"])
        ax.grid(True, alpha=0.3)

    g.axes.flat[0].set_ylabel(PLOT_CFG["ylabel"])

    # --- Legend outside figure ---
    if PLOT_CFG["legend_out"]:
        handles, labels = g.axes.flat[0].get_legend_handles_labels()

        g.fig.legend(
            handles, labels,
            loc=PLOT_CFG["legend_loc"],
            bbox_to_anchor=PLOT_CFG["legend_bbox"],
            ncol=PLOT_CFG["legend_ncol"],
            fontsize=PLOT_CFG["legend_fontsize"],
            frameon=True,
        )
        g.fig.subplots_adjust(right=PLOT_CFG["right_pad"])

    # Remove inner legend
    for ax in g.axes.flat:
        leg = ax.get_legend()
        if leg:
            leg.remove()

    out = outfile or PLOT_CFG["outfile"]
    g.fig.savefig(out, dpi=PLOT_CFG["dpi"], bbox_inches="tight")
    print(f"Saved figure as: {out}")
    plt.show()

# =============================================================================
# 7) Execution
# =============================================================================
if __name__ == "__main__":
    setup_style()
    stats_df = load_stats("../probe_results.csv")

    # Display only 1% noise model
    models_to_display = ["1% noise"]

    plot_with_std(stats_df, models_to_display)

In [ ]:
# bars_acc_icku_last.py
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

CSV_PATH   = "../probe_results.csv"
METRIC_COL = "em/multi_ood_in_ctx"   # ACC_ICKU

# Prefix-based model selection: find models matching each noise level
def find_model_by_prefix(df, prefix):
    """Find the first model name matching the given prefix."""
    matches = [m for m in df["model"].unique() if m.startswith(prefix)]
    return matches[0] if matches else None

# Load data
df = pd.read_csv(CSV_PATH)

# Discover model names by prefix
model_0pct  = find_model_by_prefix(df, "sec31_multi")
model_1pct  = find_model_by_prefix(df, "sec32_noise001")
model_5pct  = find_model_by_prefix(df, "sec32_noise005")
model_10pct = find_model_by_prefix(df, "sec32_noise010")

MODEL_IDS = [m for m in [model_0pct, model_1pct, model_5pct, model_10pct] if m is not None]
MODEL_LABELS = {}
if model_0pct:  MODEL_LABELS[model_0pct]  = "0%"
if model_1pct:  MODEL_LABELS[model_1pct]  = "1%"
if model_5pct:  MODEL_LABELS[model_5pct]  = "5%"
if model_10pct: MODEL_LABELS[model_10pct] = "10%"

ORDER = ["0%", "1%", "5%", "10%"]

sns.set_theme(style="whitegrid", rc={
    "figure.dpi": 150,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

# Percent scale correction
if METRIC_COL in df.columns and df[METRIC_COL].max() <= 1.0:
    df[METRIC_COL] *= 100.0

# Extract last checkpoint per model (single run, no seed column)
rows = []
for mid in MODEL_IDS:
    sub = df[df["model"] == mid].copy()
    if sub.empty:
        continue
    last_step = sub["step"].max()
    sub_last  = sub[sub["step"] == last_step][["model", METRIC_COL]].copy()
    sub_last["label"] = MODEL_LABELS.get(mid, mid)
    rows.append(sub_last)

if rows:
    last_df = pd.concat(rows, ignore_index=True)
    last_df = last_df.dropna(subset=[METRIC_COL])
    last_df["label"] = pd.Categorical(last_df["label"], categories=ORDER, ordered=True)
else:
    last_df = pd.DataFrame(columns=["model", METRIC_COL, "label"])

# Summary output (optional)
if not last_df.empty:
    summary = (
        last_df.groupby("label", as_index=False)[METRIC_COL]
               .agg(mean="mean", n="count")
               .sort_values("label")
    )
    print(summary.round(2))
else:
    print("No data for bar chart.")

# Plotting
fig, ax = plt.subplots(figsize=(2.5, 2.5))
sns.barplot(
    data=last_df,
    x="label", y=METRIC_COL,
    order=ORDER,
    estimator=np.mean,
    errorbar=None,     # single run, no error bars
    capsize=0.2,
    width=0.7,
    ax=ax
)
ax.set_xlabel("Noise")
ax.set_ylabel(r'$\mathrm{Acc}_{\mathrm{ICKU}}$' + "(%)")
# ax.set_title(r'$\mathrm{Acc}_{\mathrm{ICKU}}$' + " at End of Training", weight="bold", style="italic")
ax.grid(True, axis="y", linestyle="--", alpha=0.5)
sns.despine(left=False, bottom=False)

plt.tight_layout()
plt.savefig("bars_acc_icku_last.pdf", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# =============================================================================
# Per-entity probability analysis: Binned prob vs Pref_PK (all attributes)
# SKIPPED: Requires fullprob_*.csv files that have not been generated yet
# for the current project. Re-enable this cell once those files are available.
# =============================================================================
print("SKIPPED: This cell requires fullprob_*.csv data not yet generated.")

In [ ]:
# =============================================================================
# Post-training prob vs Pref_PK analysis (Scenario 1 & 2)
# SKIPPED: Requires fullprob_*.csv files that have not been generated yet
# for the current project. Re-enable this cell once those files are available.
# =============================================================================
print("SKIPPED: This cell requires fullprob_*.csv data not yet generated.")